In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
RAW_FOLDER = Path("../data/raw")
DATA_PATH = next(RAW_FOLDER.glob("*.xlsx"))

df_raw = pd.read_excel(
    DATA_PATH,
    header=1,
    engine="openpyxl"
)

df = df_raw.copy()

df.head()

,N° du client,Statut du client,Âge du client,Genre du client,Nb de personnes à charge,Niveau de diplôme,Statut marital,Catégorie du revenu annuel,Type de carte,Durée d'engagement en mois,Nb de mois inactif,Nb d'interactions,Montant crédit renouvellé,Nb de transactions,Utilisation moyenne de la carte
0,768805383,Client actuel,45,M,3,Lycée (équivalent baccalauréat),Marié(e),$60K - $80K,Blue,39,1,3,777,42,0.061
1,818770008,Client actuel,49,F,5,Licence,Célibataire,Moins de $40K,Blue,44,1,2,864,33,0.105
2,713982108,Client actuel,51,M,3,Licence,Marié(e),$80K - $120K,Blue,36,1,0,0,20,0.000
3,769911858,Client actuel,40,F,4,Lycée (équivalent baccalauréat),Non connu,Moins de $40K,Blue,34,4,1,2517,20,0.760
4,709106358,Client actuel,40,M,3,Sans diplôme,Marié(e),$60K - $80K,Blue,21,1,0,0,28,0.000


# Supprimons les espaces cachées 

In [3]:
df.columns = (
    df.columns
    .astype(str)
    .str.strip()
)

df.columns.tolist()

['N° du client',
 'Statut du client',
 'Âge du client',
 'Genre du client',
 'Nb de personnes à charge',
 'Niveau de diplôme',
 'Statut marital',
 'Catégorie du revenu annuel',
 'Type de carte',
 "Durée d'engagement en mois",
 'Nb de mois inactif',
 "Nb d'interactions",
 'Montant crédit renouvellé',
 'Nb de transactions',
 'Utilisation moyenne de la carte']

# Nettoyant les colones textuelles 

In [4]:
colonnes_texte = df.select_dtypes(include="object").columns

for colonne in colonnes_texte:
    df[colonne] = df[colonne].str.strip()

C:\Users\fayel\AppData\Local\Temp\ipykernel_17156\2351925481.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  colonnes_texte = df.select_dtypes(include="object").columns


# Verifions les catégories apres nettoyage 

In [5]:
for colonne in colonnes_texte:
    print(f"\n--- {colonne} ---")
    print(df[colonne].value_counts(dropna=False))


--- Statut du client ---
Statut du client
Client actuel    8491
Client perdu     1636
Name: count, dtype: int64

--- Genre du client ---
Genre du client
F    5358
M    4769
Name: count, dtype: int64

--- Niveau de diplôme ---
Niveau de diplôme
Licence                            3128
Lycée (équivalent baccalauréat)    2013
Non connu                          1519
Sans diplôme                       1487
Niveau Bac+2                       1013
Master                              516
Doctorat                            451
Name: count, dtype: int64

--- Statut marital ---
Statut marital
Marié(e)       4547
Célibataire    4083
Non connu       749
Divorcé(e)      748
Name: count, dtype: int64

--- Catégorie du revenu annuel ---
Catégorie du revenu annuel
Moins de $40K    3183
$40K - $60K      1961
$60K - $80K      1577
$80K - $120K     1569
Non connu        1110
$120K +           727
Name: count, dtype: int64

--- Type de carte ---
Type de carte
Blue        9436
Silver       555
Gold        

# Verifions les doublons et les types numériques 

In [8]:
print("Doublons complets :", df.duplicated().sum())
print(
    "Identifiants clients dupliqués :",
    df["N° du client"].duplicated().sum()
)
df.dtypes

Doublons complets : 0
Identifiants clients dupliqués : 0


N° du client                         int64
Statut du client                       str
Âge du client                        int64
Genre du client                        str
Nb de personnes à charge             int64
Niveau de diplôme                      str
Statut marital                         str
Catégorie du revenu annuel             str
Type de carte                          str
Durée d'engagement en mois           int64
Nb de mois inactif                   int64
Nb d'interactions                    int64
Montant crédit renouvellé            int64
Nb de transactions                   int64
Utilisation moyenne de la carte    float64
dtype: object

#  Verifions les Valeurs impossibles

In [11]:
controles = {
    "Âge négatif": (df["Âge du client"] < 0).sum(),
    "Personnes à charge négatives":
        (df["Nb de personnes à charge"] < 0).sum(),
    "Durée d'engagement négative":
        (df["Durée d'engagement en mois"] < 0).sum(),
    "Mois d'inactivité négatifs":
        (df["Nb de mois inactif"] < 0).sum(),
    "Interactions négatives":
        (df["Nb d'interactions"] < 0).sum(),
    "Crédit renouvelé négatif":
        (df["Montant crédit renouvellé"] < 0).sum(),
    "Transactions négatives":
        (df["Nb de transactions"] < 0).sum(),
    "Utilisation moyenne négative":
        (df["Utilisation moyenne de la carte"] < 0).sum()
}

pd.Series(controles)

Âge négatif                     0
Personnes à charge négatives    0
Durée d'engagement négative     0
Mois d'inactivité négatifs      0
Interactions négatives          0
Crédit renouvelé négatif        0
Transactions négatives          0
Utilisation moyenne négative    0
dtype: int64

# Exportons les données nettoyées 

In [13]:
OUTPUT_PATH = Path(
    "../data/processed/primero_bank_clients_nettoyes.csv"
)

df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print(f"Fichier exporté vers : {OUTPUT_PATH.resolve()}")

Fichier exporté vers : C:\Users\fayel\OneDrive\github\data\data-analyse-clients-banque\data\processed\primero_bank_clients_nettoyes.csv
